<a href="https://colab.research.google.com/github/princessnaya1234/Context-Aware-Audit-Tool/blob/main/Context_Aware_Audit_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Context Awareness Audit Tool

This notebook installs dependencies and launches the Gradio prototype for the paper Rethinking Context-Aware Computing Using Intersectionality.

## What this tool demonstrates

**Context Representation:** combines multiple context variables into an intersectional context profile.

**Context Evaluation:** runs a Chi-Square audit to test whether outcomes differ by context profile.

**Context Adaptation Logic:** generates rule-based adaptation recommendations when lower-performing contexts are detected.

In [ ]:
!pip install -q gradio pyreadstat openpyxl xlrd scipy pandas matplotlib

# ==========================================
# Intersectional Context Awareness Audit Tool
# Highly Interactive & Dynamically Synced UI
# ==========================================

import os
import math
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from scipy.stats import chi2_contingency, fisher_exact

import nest_asyncio
nest_asyncio.apply()


# ---------------------------------------------------
# Final-study configuration from the completed paper
# ---------------------------------------------------
ANALYSIS_SPECS = [
    {"name": "RQ1 • Gender × Digital Payment Adoption", "rq": "RQ1", "context_var": "gender_bin", "outcome_var": "digital_payment_adoption", "family": "gender"},
    {"name": "RQ1 • Income × Digital Payment Adoption", "rq": "RQ1", "context_var": "income_bin", "outcome_var": "digital_payment_adoption", "family": "income"},
    {"name": "RQ2 • Rural × Internet Adoption", "rq": "RQ2", "context_var": "rural_bin", "outcome_var": "internet_adoption", "family": "rural"},
    {"name": "RQ2 • Income × Internet Adoption", "rq": "RQ2", "context_var": "income_bin", "outcome_var": "internet_adoption", "family": "income"},
    {"name": "RQ3 • Age × Mobile Money Adoption", "rq": "RQ3", "context_var": "age_bin", "outcome_var": "mobile_money_adoption", "family": "age"},
    {"name": "RQ3 • Gender × Mobile Money Adoption", "rq": "RQ3", "context_var": "gender_bin", "outcome_var": "mobile_money_adoption", "family": "gender"},
    {"name": "RQ3 • Income × Mobile Money Adoption", "rq": "RQ3", "context_var": "income_bin", "outcome_var": "mobile_money_adoption", "family": "income"},
]

RECODE_RULES = {
    "digital_payment_adoption": {"raw": "digital_payment_pct", "threshold": 52.761332},
    "internet_adoption": {"raw": "internet_pct", "threshold": 62.856807},
    "mobile_money_adoption": {"raw": "mobile_money_pct", "threshold": 21.972201},
}

COLUMN_ALIASES = {
    "subgroup_type": ["subgroup_type", "subgroup family", "subgroup_family", "group_type"],
    "subgroup_label": ["subgroup_label", "subgroup label", "group_label", "label"],
    "gender_bin": ["gender_bin", "gender", "gender binary", "sex_bin", "sex"],
    "income_bin": ["income_bin", "income", "income binary", "income_level_bin"],
    "rural_bin": ["rural_bin", "rural", "rural_urban", "rural urban", "geography_bin", "urban_rural_bin"],
    "age_bin": ["age_bin", "age", "age group", "age_group_bin"],
    "digital_payment_pct": ["digital_payment_pct", "digital payment pct", "digital_payments_pct", "digital_payment_percentage"],
    "internet_pct": ["internet_pct", "internet pct", "internet_percentage"],
    "mobile_money_pct": ["mobile_money_pct", "mobile money pct", "mobile_money_percentage"],
    "digital_payment_adoption": ["digital_payment_adoption", "digital payment adoption", "digital_payments_adoption"],
    "internet_adoption": ["internet_adoption", "internet adoption"],
    "mobile_money_adoption": ["mobile_money_adoption", "mobile money adoption"],
}

CONTEXT_LABELS = {
    "gender_bin": {0: "Male", 1: "Female"},
    "income_bin": {0: "Richest 60%", 1: "Poorest 40%"},
    "rural_bin": {0: "Urban", 1: "Rural"},
    "age_bin": {0: "25+", 1: "15–24"},
}

OUTCOME_LABELS = {0: "Low Adoption", 1: "High Adoption"}

ADAPTATION_LIBRARY = {
    ("income_bin", "digital_payment_adoption"): ["Trigger a lighter-weight digital payment flow for lower-resource contexts.", "Prioritize fee transparency, assisted onboarding, and retry-safe transactions.", "Reduce hidden friction that assumes stable access, confidence, or uninterrupted sessions."],
    ("gender_bin", "digital_payment_adoption"): ["Audit whether onboarding, verification, or support pathways assume a default user identity.", "Check for asymmetric friction in recovery, trust, and help-seeking flows.", "Ensure adaptation logic differentiates context without stereotyping users."],
    ("rural_bin", "internet_adoption"): ["Provide low-connectivity and offline-friendly pathways for infrastructure-constrained contexts.", "Reduce dependency on always-on broadband assumptions.", "Support caching, resumable tasks, and lightweight interfaces."],
    ("income_bin", "internet_adoption"): ["Use data-saving defaults and compressed content delivery in lower-resource contexts.", "Avoid workflows that assume sustained connectivity or high-data availability.", "Add recovery mechanisms for interrupted sessions."],
    ("age_bin", "mobile_money_adoption"): ["Adjust onboarding complexity and explanation depth for the lower-performing age-coded context.", "Reduce ambiguity in financial actions with clearer step feedback.", "Support simpler, confidence-building interaction flows."],
    ("gender_bin", "mobile_money_adoption"): ["Audit whether safety, support, and trust assumptions differ across gender-coded contexts.", "Review friction in verification and account recovery pathways.", "Avoid treating one identity-coded pathway as the default interaction model."],
    ("income_bin", "mobile_money_adoption"): ["Design for constrained devices, constrained data, and interrupted sessions.", "Reduce hidden cost assumptions in the transaction experience.", "Make recovery after failed or incomplete actions more transparent."],
}

# --------------------------
# Utility / preprocessing
# --------------------------
def normalize_text(value: Any) -> str:
    return str(value).strip().lower().replace("_", " ").replace("-", " ")

def safe_read_file(file_obj) -> pd.DataFrame:
    if file_obj is None:
        raise gr.Error("Please upload your cleaned subgroup-level file (.csv, .xlsx, .xls, or .sav).")
    path = file_obj if isinstance(file_obj, str) else getattr(file_obj, "name", None)
    if not path:
        raise gr.Error("Could not access the uploaded file path.")
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv": return pd.read_csv(path)
        elif ext == ".xlsx": return pd.read_excel(path, engine="openpyxl")
        elif ext == ".xls": return pd.read_excel(path, engine="xlrd")
        elif ext == ".sav": return pd.read_spss(path)
        else: raise gr.Error(f"Unsupported file format: {ext}")
    except Exception as e:
        raise gr.Error(f"Error reading file: {e}")

def sanitize_df(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()
    clean.columns = [str(c).strip() for c in clean.columns]
    return clean

def resolve_columns(df: pd.DataFrame) -> Dict[str, str]:
    normalized_lookup = {normalize_text(col): col for col in df.columns}
    resolved = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            alias_norm = normalize_text(alias)
            if alias_norm in normalized_lookup:
                resolved[canonical] = normalized_lookup[alias_norm]
                break
    return resolved

def ensure_adoption_columns(df: pd.DataFrame, mapping: Dict[str, str]):
    notes = []
    work = df.copy()
    for adoption_col, spec in RECODE_RULES.items():
        if adoption_col not in mapping:
            raw_col = mapping.get(spec["raw"])
            threshold = spec["threshold"]
            if raw_col:
                work[adoption_col] = pd.to_numeric(work[raw_col], errors="coerce").apply(
                    lambda x: 1 if pd.notna(x) and x >= threshold else (0 if pd.notna(x) else np.nan)
                )
                mapping[adoption_col] = adoption_col
                notes.append(f"Created `{adoption_col}` from `{raw_col}` using pooled-median threshold `{threshold}`.")
    return work, mapping, notes

def validate_required_columns(mapping: Dict[str, str]):
    required = ["gender_bin", "income_bin", "rural_bin", "age_bin", "digital_payment_adoption", "internet_adoption", "mobile_money_adoption"]
    missing = [c for c in required if c not in mapping]
    if missing:
        raise gr.Error("Missing required columns:\n\n" + ", ".join(missing))

def coerce_binary_label(series: pd.Series, kind: str) -> pd.Series:
    label_map = CONTEXT_LABELS.get(kind, OUTCOME_LABELS)
    def convert(val):
        if pd.isna(val): return np.nan
        if isinstance(val, str):
            txt = normalize_text(val)
            inverse = {normalize_text(v): v for v in label_map.values()}
            if txt in inverse: return inverse[txt]
            if kind not in CONTEXT_LABELS:
                if txt in {"0", "low", "low adoption", "below median", "not adopted"}: return "Low Adoption"
                if txt in {"1", "high", "high adoption", "at or above median", "adopted"}: return "High Adoption"
            try: return label_map.get(int(float(val)), str(val))
            except: return str(val)
        if isinstance(val, (int, np.integer, float, np.floating)):
            if pd.isna(val): return np.nan
            return label_map.get(int(val), str(val))
        return str(val)
    return series.apply(convert)

def infer_family_filter(series: pd.Series, family: str) -> pd.Series:
    family_map = {"gender": ["gender", "sex"], "income": ["income"], "rural": ["rural", "urban", "rural urban", "geography", "location"], "age": ["age"]}
    keys = family_map.get(family, [family])
    return series.astype(str).apply(lambda x: any(k in normalize_text(x) for k in keys))

def get_context_order(context_var: str) -> List[str]:
    return [CONTEXT_LABELS[context_var][k] for k in sorted(CONTEXT_LABELS[context_var].keys())]

def get_outcome_order() -> List[str]:
    return [OUTCOME_LABELS[0], OUTCOME_LABELS[1]]

def format_p(p: float) -> str:
    if pd.isna(p): return "NA"
    if p < 0.001: return "< .001"
    return f"= {p:.3f}".replace("0.", ".")

def signed_phi_from_table(table: pd.DataFrame, chi2: float, n: int) -> float:
    if table.shape != (2, 2) or n == 0: return np.nan
    a, b = table.iloc[0, 0], table.iloc[0, 1]
    c, d = table.iloc[1, 0], table.iloc[1, 1]
    sign = np.sign((a * d) - (b * c))
    return float(sign * math.sqrt(chi2 / n))

def cramers_v(chi2: float, n: int, rows: int, cols: int) -> float:
    if n == 0: return np.nan
    denom = n * max(1, min(rows - 1, cols - 1))
    return float(math.sqrt(chi2 / denom)) if denom > 0 else np.nan

# --------------------------
# Plotting
# --------------------------
def build_clustered_bar(crosstab_summary: pd.DataFrame, analysis_name: str):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    categories = crosstab_summary["Category"].tolist()
    low_vals = crosstab_summary["Low Adoption n"].tolist()
    high_vals = crosstab_summary["High Adoption n"].tolist()

    x = np.arange(len(categories))
    width = 0.35

    low_color = "#6366F1"   # Indigo
    high_color = "#14B8A6"  # Teal

    ax.bar(x - width / 2, low_vals, width, label="Low Adoption", color=low_color, edgecolor="none", alpha=0.9)
    ax.bar(x + width / 2, high_vals, width, label="High Adoption", color=high_color, edgecolor="none", alpha=0.9)

    ax.set_title(analysis_name, fontweight="bold", color="#1E293B")
    ax.set_xlabel("Context Category", fontweight="semibold")
    ax.set_ylabel("Count", fontweight="semibold")
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#E2E8F0", linestyle="dashed")

    plt.tight_layout()
    return fig

# --------------------------
# Core analysis logic
# --------------------------
def analyze_pair(df: pd.DataFrame, mapping: Dict[str, str], spec: Dict[str, str]) -> Dict[str, Any]:
    context_col = mapping[spec["context_var"]]
    outcome_col = mapping[spec["outcome_var"]]
    subgroup_col = mapping.get("subgroup_type")
    total_n = len(df)
    work = df.copy()

    if subgroup_col:
        family_mask = infer_family_filter(work[subgroup_col], spec["family"])
        if family_mask.any(): work = work.loc[family_mask].copy()

    valid = work[[context_col, outcome_col]].dropna().copy()
    valid_n = len(valid)
    missing_n = total_n - valid_n

    if valid_n == 0:
        raise gr.Error(f"No valid rows found for {spec['name']}.")

    valid["context_label"] = coerce_binary_label(valid[context_col], spec["context_var"])
    valid["outcome_label"] = coerce_binary_label(valid[outcome_col], "outcome")
    valid = valid.dropna(subset=["context_label", "outcome_label"]).copy()

    context_order = get_context_order(spec["context_var"])
    outcome_order = get_outcome_order()
    contingency = pd.crosstab(valid["context_label"], valid["outcome_label"]).reindex(index=context_order, columns=outcome_order, fill_value=0)

    chi2, p_value, dof = np.nan, np.nan, np.nan
    expected_df = pd.DataFrame(0, index=contingency.index, columns=contingency.columns)
    sparse_cells = contingency.size
    min_expected = 0.0
    all_sparse = True

    try:
        chi2, p_value, dof, expected_arr = chi2_contingency(contingency, correction=False)
        expected_df = pd.DataFrame(expected_arr, index=contingency.index, columns=contingency.columns)
        sparse_cells = int((expected_df < 5).sum().sum())
        min_expected = float(expected_df.min().min())
        all_sparse = sparse_cells == expected_df.size
    except ValueError as e:
        pass

    fisher_two, fisher_less, fisher_greater, odds_ratio = np.nan, np.nan, np.nan, np.nan
    if contingency.shape == (2, 2):
        odds_ratio, fisher_two = fisher_exact(contingency.values, alternative="two-sided")
        _, fisher_less = fisher_exact(contingency.values, alternative="less")
        _, fisher_greater = fisher_exact(contingency.values, alternative="greater")

    n = int(contingency.values.sum())
    phi = signed_phi_from_table(contingency, chi2, n)
    cv = cramers_v(chi2, n, contingency.shape[0], contingency.shape[1])
    row_totals = contingency.sum(axis=1)

    summary_rows = []
    for category in contingency.index:
        low_n = int(contingency.loc[category, "Low Adoption"])
        high_n = int(contingency.loc[category, "High Adoption"])
        total = int(row_totals.loc[category])
        summary_rows.append({
            "Context Variable": spec["context_var"],
            "Category": category,
            "Low Adoption n": low_n,
            "Low Adoption %": round((low_n / total * 100) if total else 0.0, 1),
            "High Adoption n": high_n,
            "High Adoption %": round((high_n / total * 100) if total else 0.0, 1),
            "Total": total,
            "High Adoption Rate": round((high_n / total) if total else 0.0, 4),
        })

    crosstab_summary = pd.DataFrame(summary_rows)
    stats_df = pd.DataFrame([{
        "Analysis": spec["name"], "Research Question": spec["rq"], "Context Variable": spec["context_var"],
        "Outcome Variable": spec["outcome_var"], "Valid N": valid_n, "Missing N": missing_n, "Total N": total_n,
        "Pearson Chi-Square": round(float(chi2), 3) if not pd.isna(chi2) else np.nan, "df": int(dof) if not pd.isna(dof) else np.nan,
        "p": round(float(p_value), 3) if not pd.isna(p_value) else np.nan,
        "Fisher Exact (2-sided)": round(float(fisher_two), 3) if not pd.isna(fisher_two) else np.nan,
        "Odds Ratio": round(float(odds_ratio), 3) if not pd.isna(odds_ratio) else np.nan,
        "Cramer's V": round(float(cv), 3) if not pd.isna(cv) else np.nan,
        "Cells < 5": sparse_cells, "Significant @ .05": bool(p_value < 0.05) if not pd.isna(p_value) else False,
    }])

    weakest = crosstab_summary.sort_values(["High Adoption Rate", "Low Adoption n"], ascending=[True, False]).iloc[0]["Category"]
    recommendations = ADAPTATION_LIBRARY.get((spec["context_var"], spec["outcome_var"]), [
        "Audit whether the system assumes one default context pathway.",
        "Reduce friction for the lower-performing context surfaced by the audit.",
    ])

    interpretation_parts = []
    if not pd.isna(chi2): interpretation_parts.append(f"**Pearson χ²({int(dof)})** = {chi2:.3f}, p {format_p(p_value)}.")
    if not pd.isna(fisher_two): interpretation_parts.append(f"**Fisher's Exact (2-sided)** {format_p(fisher_two)}.")
    if not pd.isna(cv): interpretation_parts.append(f"**Cramer's V** = {cv:.3f}.")
    if sparse_cells > 0 and not pd.isna(chi2): interpretation_parts.append(f"⚠️ *Expected-count caution:* {sparse_cells} cell(s) fell below 5.")
    interpretation_parts.append(f"👉 The lowest-performing context category in this audit was **{weakest}**.")

    return {
        "spec": spec, "valid_n": valid_n, "missing_n": missing_n, "total_n": total_n,
        "contingency": contingency.reset_index().rename(columns={"context_label": "Category"}),
        "expected": expected_df.reset_index().rename(columns={"context_label": "Category"}),
        "crosstab_summary": crosstab_summary, "stats": stats_df,
        "interpretation": "\n\n".join(interpretation_parts),
        "recommendations_md": "\n".join([f"- {x}" for x in recommendations]),
        "figure": build_clustered_bar(crosstab_summary, spec["name"]),
    }

def build_case_processing(results: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame([{"Analysis": r["spec"]["name"], "Valid N": r["valid_n"], "Missing N": r["missing_n"], "Total N": r["total_n"]} for r in results])

def build_overall_summary(results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        s = r["stats"].iloc[0]
        rows.append({
            "Focal Test": s["Analysis"], "Pearson Chi-Square": s["Pearson Chi-Square"],
            "p": s["p"], "Cramer's V": s["Cramer's V"], "Significant @ .05": s["Significant @ .05"]
        })
    return pd.DataFrame(rows)

# --------------------------
# Dynamic Report Builder
# --------------------------
def build_synced_audit_report(result: Dict[str, Any]) -> str:
    """Dynamically creates the Audit Summary based on the selected analysis."""
    if not result:
        return "Please run the audit and select an analysis to view its summary."

    spec = result["spec"]

    report = f"""
### 📊 Audit Summary: {spec['name']}

**Context Variable:** `{spec['context_var']}` | **Outcome Variable:** `{spec['outcome_var']}`

#### Statistical Interpretation
{result['interpretation']}

---

#### 🛠 Context Adaptation Recommendations
Based on the intersectional audit logic, the following framework adaptations are recommended to reduce systemic friction:
{result['recommendations_md']}
    """
    return report.strip()

# --------------------------
# Run all final-study audits
# --------------------------
def run_final_audit(file_obj):
    df = sanitize_df(safe_read_file(file_obj))
    mapping = resolve_columns(df)
    df, mapping, recode_notes = ensure_adoption_columns(df, mapping)
    validate_required_columns(mapping)

    results = [analyze_pair(df, mapping, spec) for spec in ANALYSIS_SPECS]

    case_df = build_case_processing(results)
    overall_df = build_overall_summary(results)
    all_crosstabs_df = pd.concat([r["crosstab_summary"].assign(Analysis=r["spec"]["name"]) for r in results], ignore_index=True)
    all_stats_df = pd.concat([r["stats"] for r in results], ignore_index=True)

    recommendations_md = "\n\n".join([f"#### {r['spec']['name']}\n{r['recommendations_md']}" for r in results])

    detail_lookup = {r["spec"]["name"]: r for r in results}
    default_key = ANALYSIS_SPECS[0]["name"]
    default_result = detail_lookup[default_key]

    # Generate the initial dynamic report based on the default key
    dynamic_report_md = build_synced_audit_report(default_result)

    status_lines = [f"✅ Audit completed for **{len(results)} focal analyses**.", f"Loaded **{len(df):,} rows**."]
    if recode_notes:
        status_lines.append("**Preprocessing applied:**")
        status_lines.extend([f"- {note}" for note in recode_notes])

    bundle = {"details": detail_lookup, "status": "\n\n".join(status_lines)}

    return (
        bundle["status"], df.head(15), case_df, overall_df, all_crosstabs_df, all_stats_df,
        recommendations_md,
        dynamic_report_md,  # Maps to the 'Audit Summary' text block
        bundle,
        gr.update(choices=list(detail_lookup.keys()), value=default_key),
        default_result["contingency"], default_result["expected"], default_result["crosstab_summary"],
        default_result["stats"], default_result["interpretation"], default_result["figure"],
    )

def show_detail(bundle, analysis_name):
    if not bundle or "details" not in bundle or not analysis_name:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), "Run the audit first.", None, "No data."

    result = bundle["details"][analysis_name]
    dynamic_report_md = build_synced_audit_report(result)

    return (
        result["contingency"], result["expected"], result["crosstab_summary"],
        result["stats"], result["interpretation"], result["figure"],
        dynamic_report_md # Pushes updated text to the synced summary tab
    )


# --------------------------
# Highly Interactive CSS
# --------------------------
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

:root {
  --primary-glow: rgba(99, 102, 241, 0.4);
  --bg-gradient: linear-gradient(135deg, #F8FAFC 0%, #E2E8F0 100%);
  --card-bg: rgba(255, 255, 255, 0.85);
  --text-main: #0F172A;
  --accent: #4F46E5;
  --accent-secondary: #14B8A6;
}

body, .gradio-container {
  background: var(--bg-gradient);
  font-family: 'Inter', sans-serif !important;
  color: var(--text-main);
}

#hero {
  background: linear-gradient(135deg, #1E293B, #0F172A);
  border-radius: 20px;
  padding: 32px;
  color: white;
  box-shadow: 0 20px 40px rgba(15, 23, 42, 0.2);
  margin-bottom: 24px;
  position: relative;
  overflow: hidden;
}

#hero::after {
  content: '';
  position: absolute;
  top: -50%; left: -50%; width: 200%; height: 200%;
  background: radial-gradient(circle, var(--primary-glow) 0%, transparent 60%);
  opacity: 0.5; pointer-events: none;
}

#hero h1 { color: #F8FAFC !important; font-weight: 700; letter-spacing: -0.02em; font-size: 2.2rem; }
#hero p { color: #94A3B8 !important; font-size: 1.1rem; line-height: 1.6; max-width: 800px; }

.ctx-card {
  background: var(--card-bg);
  backdrop-filter: blur(12px);
  border: 1px solid rgba(255, 255, 255, 0.5);
  border-radius: 16px;
  box-shadow: 0 10px 25px rgba(0,0,0,0.05);
  padding: 24px;
  transition: transform 0.2s ease, box-shadow 0.2s ease;
}

/* Interactive HTML Grid Cards */
.op-grid {
  display: flex;
  flex-direction: column;
  gap: 16px;
}

.op-card {
  background: white;
  border-radius: 12px;
  padding: 16px 20px;
  border-left: 5px solid var(--accent);
  box-shadow: 0 4px 6px rgba(0,0,0,0.02);
  transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
  cursor: default;
}

.op-card:nth-child(2) { border-left-color: var(--accent-secondary); }
.op-card:nth-child(3) { border-left-color: #F59E0B; }

.op-card:hover {
  transform: translateX(6px) scale(1.01);
  box-shadow: 0 12px 20px rgba(0,0,0,0.08);
}

.op-card h3 {
  margin: 0 0 8px 0;
  font-size: 1.05rem;
  color: var(--text-main);
}

.op-card p {
  margin: 0;
  font-size: 0.9rem;
  color: #64748B;
  line-height: 1.5;
}

button.primary {
  background: linear-gradient(135deg, var(--accent), #818CF8) !important;
  border: none !important;
  font-weight: 600 !important;
  letter-spacing: 0.5px;
  transition: all 0.2s ease !important;
}

button.primary:hover {
  transform: translateY(-2px);
  box-shadow: 0 8px 20px var(--primary-glow) !important;
}
"""

# --------------------------
# Interactive HTML Layout
# --------------------------
OPERATIONAL_HTML = """
<div class="op-grid">
    <div class="op-card">
        <h3>📊 Context Representation</h3>
        <p>Operationalizes binary social context fields established in the paper to evaluate systemic adoption behaviors.</p>
    </div>
    <div class="op-card">
        <h3>🧮 Context Evaluation</h3>
        <p>Executes deep statistical reviews utilizing 2×2 crosstabs, Pearson Chi-Square, Fisher's Exact Test, and Cramer's V.</p>
    </div>
    <div class="op-card">
        <h3>⚙️ Adaptation Logic</h3>
        <p>Generates targeted, rule-based recommendations when statistically lower-performing contexts are surfaced by the audit.</p>
    </div>
</div>
"""

# --------------------------
# Build Gradio app
# --------------------------
with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Default(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:
    audit_state = gr.State({})

    gr.Markdown("""
    <div id="hero">
        <h1>Intersectional Context Awareness Audit Tool</h1>
        <p>
            An interactive implementation of the subgroup-level audit design. This tool evaluates
            the seven focal context–outcome analyses, dynamically calculates cross-tabulation metrics,
            and maps quantitative deficits to structural adaptation rules.
        </p>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, elem_classes=["ctx-card"]):
            file_input = gr.File(label="Upload Dataset (.csv, .xlsx, .xls, .sav)")
            run_btn = gr.Button("Execute Audit", variant="primary")
            status_md = gr.Markdown("Ready for data ingestion.")
            analysis_selector = gr.Dropdown(choices=[], label="Select Focal Analysis", interactive=True)

        with gr.Column(scale=1, elem_classes=["ctx-card"]):
            gr.Markdown("### Architectural Overview")
            gr.HTML(OPERATIONAL_HTML) # Beautiful interactive HTML cards replace static Markdown

    with gr.Tabs():
        with gr.TabItem("Audit Summary"):
            report_md = gr.Markdown("Select an analysis from the dropdown to dynamically generate the audit summary.")

        with gr.TabItem("Detailed Analysis"):
            detail_note = gr.Markdown("Inspect granular statistical outputs.")
            with gr.Row():
                with gr.Column(scale=1):
                    detail_cont = gr.Dataframe(label="Contingency Table")
                    detail_exp = gr.Dataframe(label="Expected Counts")
                    detail_rates = gr.Dataframe(label="Row Percentages & Adoption Rates")
                with gr.Column(scale=1):
                    detail_plot = gr.Plot(label="Clustered Adoption Bar Chart")
            detail_stats = gr.Dataframe(label="Full Statistical Metrics")

        with gr.TabItem("Data Preview"):
            preview_df = gr.Dataframe(label="Uploaded Dataset Segment")

        with gr.TabItem("Case Processing"):
            case_df = gr.Dataframe(label="Validation & Missingness")

        with gr.TabItem("Overall Results"):
            overall_df = gr.Dataframe(label="Macro Summary of Findings")
            all_crosstabs_df = gr.Dataframe(label="Aggregated Crosstabs")
            all_stats_df = gr.Dataframe(label="Aggregated Test Results")

        with gr.TabItem("Global Adaptation Library"):
            recommendations_md = gr.Markdown()

    # Execution Flow
    run_btn.click(
        fn=run_final_audit,
        inputs=[file_input],
        outputs=[
            status_md, preview_df, case_df, overall_df, all_crosstabs_df, all_stats_df,
            recommendations_md, report_md, audit_state, analysis_selector,
            detail_cont, detail_exp, detail_rates, detail_stats, detail_note, detail_plot,
        ],
    )

    # Reactive Sync Flow
    analysis_selector.change(
        fn=show_detail,
        inputs=[audit_state, analysis_selector],
        outputs=[
            detail_cont, detail_exp, detail_rates, detail_stats,
            detail_note, detail_plot, report_md # Dynamically updates the Audit Summary tab
        ],
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.4 MB/s eta 0:00:00


/tmp/ipykernel_8026/3523448824.py:542: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Default(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:
/tmp/ipykernel_8026/3523448824.py:542: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Default(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://103fc577a888eba776.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
